# Module 1 - ICSI Stress Test

Measure full Module 1 latency on 3-minute windows from `argmaxinc/icsi-meetings`.

This notebook uses a dedicated cache directory for ICSI to avoid incompatible stale metadata in the default Hugging Face cache.

In [1]:
import os
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from datasets import DownloadMode, load_dataset
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / 'backend' / '.env', override=False)

from backend.app.module1 import process_audio_chunk
from backend.app.module1 import asr as asr_module
from backend.app.module1 import diarization as diarization_module

In [2]:
WINDOW_SEC = 180
MEETING_COUNT = 2
LATENCY_THRESHOLD_SEC = 15.0
ICSI_CACHE_DIR = PROJECT_ROOT / 'data' / 'hf_cache' / 'icsi_meetings'
ICSI_CACHE_DIR.mkdir(parents=True, exist_ok=True)

ds = load_dataset(
    'argmaxinc/icsi-meetings',
    split='test',
    cache_dir=str(ICSI_CACHE_DIR),
    download_mode=DownloadMode.FORCE_REDOWNLOAD,
).select(range(MEETING_COUNT))
ds

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/75 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'timestamps_start', 'timestamps_end', 'speakers'],
    num_rows: 2
})

In [3]:
def iter_windows(audio_array, sample_rate, window_sec=WINDOW_SEC):
    samples_per_window = sample_rate * window_sec
    total_samples = len(audio_array)
    for start_idx in range(0, total_samples, samples_per_window):
        end_idx = min(start_idx + samples_per_window, total_samples)
        yield start_idx / sample_rate, end_idx / sample_rate, audio_array[start_idx:end_idx]


def evaluate_model(model_size: str) -> pd.DataFrame:
    os.environ['WHISPER_MODEL_SIZE'] = model_size
    os.environ['DIARIZATION_ENABLED'] = 'true'
    asr_module._get_whisper_model.cache_clear()
    diarization_module._get_diarization_pipeline.cache_clear()

    rows = []
    for meeting_idx, row in enumerate(ds):
        audio = row['audio']
        audio_array = np.asarray(audio['array'])
        sample_rate = int(audio['sampling_rate'])

        for chunk_id, (start_sec, end_sec, window_audio) in enumerate(iter_windows(audio_array, sample_rate), start=1):
            with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                sf.write(tmp.name, window_audio, sample_rate)
                started = time.perf_counter()
                transcript = process_audio_chunk(
                    wav_path=tmp.name,
                    meeting_id=f'icsi-{meeting_idx}',
                    chunk_id=chunk_id,
                    chunk_start=float(start_sec),
                    chunk_end=float(end_sec),
                )
                elapsed = time.perf_counter() - started
            Path(tmp.name).unlink(missing_ok=True)

            rows.append({
                'model': model_size,
                'meeting_index': meeting_idx,
                'chunk_id': chunk_id,
                'chunk_duration_sec': end_sec - start_sec,
                'processing_time_sec': elapsed,
                'within_15s_budget': elapsed < LATENCY_THRESHOLD_SEC,
                'display_utterances': len(transcript.display_utterances),
                'llm_utterances': len(transcript.llm_utterances),
            })
    return pd.DataFrame(rows)

In [ ]:
base_df = evaluate_model('base')
small_df = evaluate_model('small')
results = pd.concat([base_df, small_df], ignore_index=True)

summary = (
    results.groupby('model')
    .agg(
        mean_processing_time_sec=('processing_time_sec', 'mean'),
        median_processing_time_sec=('processing_time_sec', 'median'),
        max_processing_time_sec=('processing_time_sec', 'max'),
        within_15s_rate=('within_15s_budget', 'mean'),
        avg_display_utterances=('display_utterances', 'mean'),
        avg_llm_utterances=('llm_utterances', 'mean'),
        num_chunks=('chunk_id', 'count'),
    )
    .reset_index()
)
summary